# Bitcoin Time Series Forecasting Comparison
This notebook implements and compares four different models for forecasting Bitcoin prices:
1. **ARIMA** (AutoRegressive Integrated Moving Average)
2. **SARIMA** (Seasonal ARIMA)
3. **Facebook Prophet**
4. **LSTM** (Long Short-Term Memory Neural Network)

The dataset used contains historical Bitcoin 'close' prices.

## 1. Environment Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import warnings

warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')

# Load dataset
df = pd.read_excel('cleaned_bitcoin_data.xlsx')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').set_index('date')
df = df.asfreq('D')

# Select target variable and handle missing values
df['close'] = df['close'].ffill()
target = df['close']

# Split into 80% train and 20% test
split_idx = int(len(target) * 0.8)
train, test = target.iloc[:split_idx], target.iloc[split_idx:]

print(f"Total samples: {len(target)}")
print(f"Training samples: {len(train)}")
print(f"Testing samples: {len(test)}")

results = {}

## 2. Model 1: ARIMA
We use `pmdarima.auto_arima` to automatically find the best (p,d,q) parameters.

In [ ]:
print("Finding best ARIMA parameters...")
model_arima = auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True)
print(f"Best ARIMA Order: {model_arima.order}")

arima_pred = model_arima.predict(n_periods=len(test))
arima_pred.index = test.index

## 3. Model 2: SARIMA
Similar to ARIMA but with seasonal components (m=7 for weekly seasonality).

In [ ]:
print("Finding best SARIMA parameters...")
model_sarima = auto_arima(train, seasonal=True, m=7, stepwise=True, suppress_warnings=True)
print(f"Best SARIMA Order: {model_sarima.order} x {model_sarima.seasonal_order}")

sarima_pred = model_sarima.predict(n_periods=len(test))
sarima_pred.index = test.index

## 4. Model 3: Facebook Prophet
Prophet requires the data to be in a specific format (ds, y).

In [ ]:
df_prophet = train.reset_index().rename(columns={'date': 'ds', 'close': 'y'})
model_prophet = Prophet()
model_prophet.fit(df_prophet)

future = model_prophet.make_future_dataframe(periods=len(test))
forecast = model_prophet.predict(future)
prophet_pred = forecast['yhat'].iloc[-len(test):]
prophet_pred.index = test.index

## 5. Model 4: LSTM
Deep learning approach using a sliding window of 60 days.

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(target.values.reshape(-1, 1))

def create_sequences(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)

window_size = 60
X, y = create_sequences(scaled_data, window_size)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model_lstm = Sequential([
    LSTM(50, return_sequences=True, input_shape=(window_size, 1)),
    LSTM(50),
    Dense(1)
])
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

lstm_pred_scaled = model_lstm.predict(X_test)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
test_actual_lstm = scaler.inverse_transform(y_test).flatten()

## 6. Evaluation & Comparison

In [ ]:
def calculate_metrics(actual, pred, name):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    mape = mean_absolute_percentage_error(actual, pred) * 100
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

results['ARIMA'] = calculate_metrics(test, arima_pred, 'ARIMA')
results['SARIMA'] = calculate_metrics(test, sarima_pred, 'SARIMA')
results['Prophet'] = calculate_metrics(test, prophet_pred, 'Prophet')
results['LSTM'] = calculate_metrics(test_actual_lstm, lstm_pred, 'LSTM')

comparison_df = pd.DataFrame(results).T
print("Model Comparison Table:")
display(comparison_df)

best_model = comparison_df['RMSE'].idxmin()
print(f"\nBest Performing Model based on RMSE: {best_model}")

## 7. Visualization

In [ ]:
plt.figure(figsize=(15, 12))

# ARIMA
plt.subplot(3, 2, 1)
plt.plot(test.index, test.values, label='Actual', color='black')
plt.plot(test.index, arima_pred, label='ARIMA', linestyle='--', color='blue')
plt.title('ARIMA Forecast')
plt.legend()

# SARIMA
plt.subplot(3, 2, 2)
plt.plot(test.index, test.values, label='Actual', color='black')
plt.plot(test.index, sarima_pred, label='SARIMA', linestyle='--', color='green')
plt.title('SARIMA Forecast')
plt.legend()

# Prophet
plt.subplot(3, 2, 3)
plt.plot(test.index, test.values, label='Actual', color='black')
plt.plot(test.index, prophet_pred, label='Prophet', linestyle='--', color='orange')
plt.title('Prophet Forecast')
plt.legend()

# LSTM
plt.subplot(3, 2, 4)
plt.plot(test_actual_lstm, label='Actual', color='black')
plt.plot(lstm_pred, label='LSTM', linestyle='--', color='red')
plt.title('LSTM Forecast')
plt.legend()

# Combined Comparison
plt.subplot(3, 1, 3)
plt.plot(test.index, test.values, label='Actual', color='black', linewidth=2)
plt.plot(test.index, arima_pred, label='ARIMA', alpha=0.7)
plt.plot(test.index, sarima_pred, label='SARIMA', alpha=0.7)
plt.plot(test.index, prophet_pred, label='Prophet', alpha=0.7)
plt.title('Model Comparison (Classical & Prophet)')
plt.legend()

plt.tight_layout()
plt.show()